In [1]:
# Uncomment and run this command and subsequent ones if they are not present on your system!
# NOTE - CUDA compatibility may be different for other systems. Check your GPU compatibility!
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126

In [2]:
# !pip install pillow

In [3]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
import time
import matplotlib.pyplot as plt

from torchvision.models import vgg16, VGG16_Weights
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from PIL import Image

## Neural Style Transfer

This is a PyTorch implementation of the method that was discussed in Shen et. Al.

In [4]:
# Standard ImageNet normalization parameters expected by VGG/MetaN backbones
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
max_size = 256 if torch.cuda.is_available() else 128  # Resize if using CPU

def load_image(image_path):
    """Loads an image, resizes it, and converts it to a normalized PyTorch tensor."""
    image = Image.open(image_path).convert("RGB")
    
    # Scale while maintaining aspect ratio
    transform = transforms.Compose([
        transforms.Resize(max_size),
        transforms.CenterCrop(max_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    # Return tensor with added batch dimension: (1, C, H, W)
    return transform(image).unsqueeze(0)

def save_image(tensor, output_path):
    """Denormalizes output tensor and saves it to an image file on disk."""
    image = tensor.clone().squeeze(0).cpu()
    
    # Reverse ImageNet normalization: x = tensor * std + mean
    for t, m, s in zip(image, IMAGENET_MEAN, IMAGENET_STD):
        t.mul_(s).add_(m)
        
    # Clamp pixel values back to valid range [0, 1]
    image = torch.clamp(image, 0, 1)
    
    # Convert tensor back to PIL Image and save
    to_pil = transforms.ToPILImage()
    pil_img = to_pil(image)
    pil_img.save(output_path)
    print(f"Successfully saved stylized image to: {output_path}")

In [5]:
IMAGENET_MEAN = torch.tensor(
    [0.485, 0.456, 0.406]
).view(3, 1, 1)

IMAGENET_STD = torch.tensor(
    [0.229, 0.224, 0.225]
).view(3, 1, 1)


def denormalize(img):
    img = img.detach().cpu()

    mean = IMAGENET_MEAN
    std = IMAGENET_STD

    img = img * std + mean
    return img.clamp(0, 1)

In [6]:
# Style Feature Extractor
class VGG16StyleFeatureExtractor(nn.Module):
    """
    Extracts multi-scale style features from pre-trained VGG-16 layers.
    """
    def __init__(self):
        super().__init__()
        # Load pre-trained VGG-16 features
        vgg = vgg16(weights=VGG16_Weights.DEFAULT).features
        
        # Slices corresponding to relu1_2, relu2_2, relu3_3, relu4_3
        self.slice1 = vgg[:4]    # up to relu1_2 (64 channels)
        self.slice2 = vgg[4:9]   # up to relu2_2 (128 channels)
        self.slice3 = vgg[9:16]  # up to relu3_3 (256 channels)
        self.slice4 = vgg[16:23] # up to relu4_3 (512 channels)
        
        # Freeze backbone parameters
        for param in self.parameters():
            param.requires_grad = False
            
    def forward(self, x):
        h1 = self.slice1(x)
        h2 = self.slice2(h1)
        h3 = self.slice3(h2)
        h4 = self.slice4(h3)
        return [h1, h2, h3, h4]

def calc_mean_std(feat, eps=1e-5):
    """
    Computes spatial channel-wise mean and std for a feature tensor (B, C, H, W).
    """
    batch_size, channels = feat.size(0), feat.size(1)
    
    # Reshape (B, C, H, W) -> (B, C, H*W)
    feat_flat = feat.view(batch_size, channels, -1)
    
    mean = feat_flat.mean(dim=2)  # (B, C)
    std = feat_flat.std(dim=2) + eps  # (B, C)
    
    return mean, std

In [7]:
# Style and Content Loss
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = vgg16(weights=VGG16_Weights.DEFAULT).features.eval()
        
        # Layer slices for perceptual evaluation
        self.slice1 = vgg[:4]    # relu1_2
        self.slice2 = vgg[4:9]   # relu2_2
        self.slice3 = vgg[9:16]  # relu3_3
        self.slice4 = vgg[16:23] # relu4_3
        
        for p in self.parameters():
            p.requires_grad = False

    def gram_matrix(self, x):
        (b, c, h, w) = x.size()
        features = x.view(b, c, h * w)
        gram = torch.bmm(features, features.transpose(1, 2))
        return gram / (c * h * w)

    def forward(self, stylized, content, style):
        # Extract features
        s1 = self.slice1(stylized)
        s2 = self.slice2(s1)
        s3 = self.slice3(s2)
        s4 = self.slice4(s3)

        with torch.no_grad():
            c1 = self.slice1(content)
            c2 = self.slice2(c1)
            c3 = self.slice3(c2) # Content representation layer

        with torch.no_grad():
            st1 = self.slice1(style)
            st2 = self.slice2(st1)
            st3 = self.slice3(st2)
            st4 = self.slice4(st3)

            gram_st1 = self.gram_matrix(st1)
            gram_st2 = self.gram_matrix(st2)
            gram_st3 = self.gram_matrix(st3)
            gram_st4 = self.gram_matrix(st4)

        # Content Loss (MSE on relu3_3 features)
        content_loss = F.mse_loss(s3, c3)

        # General normalization
        def normalized_gram_loss(generated, target, eps=1e-6, max_loss=4.0):
            numerator = F.mse_loss(generated, target)
        
            denominator = target.pow(2).mean().detach() + eps
            loss = numerator / denominator
            return torch.clamp(loss, max=max_loss)
        
        style_loss1 = normalized_gram_loss(self.gram_matrix(s1), gram_st1)
        style_loss2 = normalized_gram_loss(self.gram_matrix(s2), gram_st2)
        style_loss3 = normalized_gram_loss(self.gram_matrix(s3), gram_st3)
        style_loss4 = normalized_gram_loss(self.gram_matrix(s4), gram_st4)
        
        style_loss = (
            style_loss1 +
            style_loss2 +
            style_loss3 +
            style_loss4
        ) / 4.0

        return content_loss, style_loss, style_loss1, style_loss2, style_loss3, style_loss4

def total_variation_loss(img):
    h_variant = torch.abs(img[:, :, :, :-1] - img[:, :, :, 1:]).mean()
    v_variant = torch.abs(img[:, :, :-1, :] - img[:, :, 1:, :]).mean()

    return h_variant + v_variant

In [8]:
def normalize_dynamic_weight(weight, target_std, eps=1e-3):
    """
    weight: [B, Cout, Cin, K, K]
    """
    B = weight.size(0)

    mean = weight.mean(dim=(1, 2, 3, 4), keepdim=True)
    centered = weight - mean
    raw_std = centered.flatten(1).std(dim=1, keepdim=True).view(B, 1, 1, 1, 1)
    # Conditional normalization
    scale = torch.clamp(raw_std, min=eps)

    return centered / scale * torch.clamp(raw_std / target_std, max=1.0) * target_std

In [9]:
def kernel_smoothness_loss(weight):
    """
    weight: [B, Cout, Cin, K, K]
    Penalizes high-frequency (checkerboard-like) differences between
    adjacent taps within each generated K x K kernel -- a TV-style loss
    applied to the conv weights themselves, not the output image.
    """
    h_diff = weight[:, :, :, :, :-1] - weight[:, :, :, :, 1:]
    v_diff = weight[:, :, :, :-1, :] - weight[:, :, :, 1:, :]
    return h_diff.abs().mean() +v_diff.abs().mean()

In [10]:
class MetaNetwork(nn.Module):
    def __init__(self, bottleneck_dim=1920):
        super().__init__()
        # Instantiate frozen VGG-16 style feature extractor
        self.vgg_extractor = VGG16StyleFeatureExtractor().eval()
        self.fc = nn.Linear(bottleneck_dim, 1792)
        self.h_norm = nn.LayerNorm(1792)
        # Linear projection heads
        self.head_down1 = nn.Linear(1792, 16 * 8 * 3 * 3)
        self.head_down2 = nn.Linear(1792, 32 * 16 * 3 * 3)
        self.head_res1_conv1 = nn.Linear(1792, 32 * 32 * 3 * 3)
        self.head_res1_conv2 = nn.Linear(1792, 32 * 32 * 3 * 3)
        self.head_res2_conv1 = nn.Linear(1792, 32 * 32 * 3 * 3)
        self.head_res2_conv2 = nn.Linear(1792, 32 * 32 * 3 * 3)
        self.head_res3_conv1 = nn.Linear(1792, 32 * 32 * 3 * 3)
        self.head_res3_conv2 = nn.Linear(1792, 32 * 32 * 3 * 3)
        self.head_res4_conv1 = nn.Linear(1792, 32 * 32 * 3 * 3)
        self.head_res4_conv2 = nn.Linear(1792, 32 * 32 * 3 * 3)
        self.head_res5_conv1 = nn.Linear(1792, 32 * 32 * 3 * 3)
        self.head_res5_conv2 = nn.Linear(1792, 32 * 32 * 3 * 3)
        self.head_up1 = nn.Linear(1792, 16 * 32 * 3 * 3)
        self.head_up2 = nn.Linear(1792, 8 * 16 * 3 * 3)
        
        self._initialize_heads() 

    def extract_style_features(self, style_img):
        """
        Extracts channel-wise mean and std from multi-layer VGG-16 activations.
        """
        self.vgg_extractor.eval()
        with torch.no_grad():
            features = self.vgg_extractor(style_img)
            
        stats = []
        for feat in features:
            mean, std = calc_mean_std(feat)
            stats.extend([mean.flatten(1), std.flatten(1)])
            
        # Concatenate channel means and stds into a single 1D feature tensor
        style_features = torch.cat(stats, dim=1) # Shape: (batch_size, 1920)
        return style_features

    @staticmethod
    def _init_meta_head(head, target_std):
        nn.init.normal_(
            head.weight,
            mean=0.0,
            std=target_std / (1792 ** 0.5)
        )
        nn.init.zeros_(head.bias)

    def _initialize_heads(self):
        def conv_std(in_ch, k=3):
            return (2.0 / (in_ch * k * k)) ** 0.5

        self._init_meta_head(self.head_down1, conv_std(8))
        self._init_meta_head(self.head_down2, conv_std(16))
        
        internal_heads = [
            self.head_res1_conv1,
            self.head_res1_conv2,
            self.head_res2_conv1,
            self.head_res2_conv2,
            self.head_res3_conv1,
            self.head_res3_conv2,
            self.head_res4_conv1,
            self.head_res4_conv2,
            self.head_res5_conv1,
            self.head_res5_conv2,
        ]
        for head in internal_heads:
            self._init_meta_head(head, conv_std(32))

        self._init_meta_head(self.head_up1, conv_std(32))
        self._init_meta_head(self.head_up2, conv_std(16))

    def forward(self, style_img):
        batch_size = style_img.size(0)
    
        style_feats = self.extract_style_features(style_img)
        h = torch.relu(self.fc(style_feats))
        h = self.h_norm(h)

        weight_scale = 0.2
    
        weights = {
            'down1_w': weight_scale * self.head_down1(h).view(batch_size, 16, 8, 3, 3),
            'down2_w': weight_scale * self.head_down2(h).view(batch_size, 32, 16, 3, 3),
            'res1_conv1_w': weight_scale * self.head_res1_conv1(h).view(batch_size, 32, 32, 3, 3),
            'res1_conv2_w': weight_scale * self.head_res1_conv2(h).view(batch_size, 32, 32, 3, 3),
            'res2_conv1_w': weight_scale * self.head_res2_conv1(h).view(batch_size, 32, 32, 3, 3),
            'res2_conv2_w': weight_scale * self.head_res2_conv2(h).view(batch_size, 32, 32, 3, 3),
            'res3_conv1_w': weight_scale * self.head_res3_conv1(h).view(batch_size, 32, 32, 3, 3),
            'res3_conv2_w': weight_scale * self.head_res3_conv2(h).view(batch_size, 32, 32, 3, 3),
            'res4_conv1_w': weight_scale * self.head_res4_conv1(h).view(batch_size, 32, 32, 3, 3),
            'res4_conv2_w': weight_scale * self.head_res4_conv2(h).view(batch_size, 32, 32, 3, 3),
            'res5_conv1_w': weight_scale * self.head_res5_conv1(h).view(batch_size, 32, 32, 3, 3),
            'res5_conv2_w': weight_scale * self.head_res5_conv2(h).view(batch_size, 32, 32, 3, 3),
            'up1_w': weight_scale * self.head_up1(h).view(batch_size, 16, 32, 3, 3),
            'up2_w': weight_scale * self.head_up2(h).view(batch_size, 8, 16, 3, 3),
        }
        # Modify the ceilings as required - these serve as a "cap" on the generated filters
        keys = ['down1_w', 'down2_w', 'up1_w', 'up2_w'] + \
               [f'res{i}_conv{j}_w' for i in range(1,6) for j in (1,2)]
        
        kernel_tv_loss = 0.0
        for k in keys:
            weights[k] = normalize_dynamic_weight(weights[k], 0.15)
            kernel_tv_loss = kernel_tv_loss + kernel_smoothness_loss(weights[k])
        kernel_tv_loss = kernel_tv_loss / len(keys)
    
        return weights

In [11]:
# Dynamic convolutional kernel
def dynamic_conv2d(x, weight, bias=None, padding=0, stride=1):
    B, C_in, H, W = x.shape
    _, C_out, _, K, _ = weight.shape

    # Treat each batch element as a separate group.
    x = x.reshape(1, B * C_in, H, W)
    weight = weight.reshape(B * C_out, C_in, K, K)

    if bias is not None:
        bias = bias.reshape(B * C_out)

    out = F.conv2d(x, weight, bias=bias, stride=stride, padding=padding, groups=B)

    return out.reshape(B, C_out, out.shape[-2], out.shape[-1])

In [12]:
class ResidualBlock(nn.Module):
    def __init__(self, channels=32):
        super().__init__()
        self.in1 = nn.InstanceNorm2d(channels, affine=False)
        self.in2 = nn.InstanceNorm2d(channels, affine=False)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, weights_prefix, generated_weights):
        # Uses weights produced by MetaNetwork
        res = dynamic_conv2d(x, generated_weights[f'{weights_prefix}_conv1_w'], padding=0)
        res = self.relu(self.in1(res))
        
        res = dynamic_conv2d(res, generated_weights[f'{weights_prefix}_conv2_w'], padding=0)
        res = self.in2(res)

        diff_h = x.shape[-2] - res.shape[-2]
        diff_w = x.shape[-1] - res.shape[-1]
        x_cropped = x[..., diff_h // 2 : x.shape[-2] - diff_h // 2,
                     diff_w // 2 : x.shape[-1] - diff_w // 2]
        
        return x_cropped + res  # Residual skip connection

In [13]:
class TransformationNetwork(nn.Module):
    """
    Image Transformation Network N(Ic; w) that processes the content image
    using dynamically generated dynamic weights w.
    """
    def __init__(self, channels=32):
        super().__init__()
        self.conv_in = nn.Conv2d(3, 8, kernel_size=9, stride=1, padding=4)
        self.conv_out = nn.Conv2d(8, 3, kernel_size=9, stride=1, padding=4)
        
        self.in_in = nn.InstanceNorm2d(8, affine=False)
        self.in_down1 = nn.InstanceNorm2d(16, affine=False)
        self.in_down2 = nn.InstanceNorm2d(32, affine=False)
        self.res1 = ResidualBlock(channels) # Sub-block inside TransformationNetwork
        self.res2 = ResidualBlock(channels)
        self.res3 = ResidualBlock(channels)
        self.res4 = ResidualBlock(channels)
        self.res5 = ResidualBlock(channels)
        self.in_up1 = nn.InstanceNorm2d(16, affine=False)
        self.in_up2 = nn.InstanceNorm2d(8, affine=False)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, content_img, generated_weights, debug=False):
        x = F.pad(content_img, (40, 40, 40, 40), mode = 'reflect')
        
        #out = dynamic_conv2d(content_img, generated_weights['conv_in_w'], padding=4)
        out = self.conv_in(x)
        out = self.relu(self.in_in(out))

        # Downscaling
        out = dynamic_conv2d(out, generated_weights['down1_w'], padding=1, stride=2)
        out = self.relu(self.in_down1(out))

        # Downscaling further
        out = dynamic_conv2d(out, generated_weights['down2_w'], padding=1, stride=2)
        out = self.relu(self.in_down2(out))
        
        # Pass features through residual block, at a 64 x 64 format
        out = self.res1(out, 'res1', generated_weights)
        out = self.res2(out, 'res2', generated_weights)
        out = self.res3(out, 'res3', generated_weights)
        out = self.res4(out, 'res4', generated_weights)
        out = self.res5(out, 'res5', generated_weights)

        # Upscaling
        out = F.interpolate(out, scale_factor=2, mode='bilinear', align_corners=False)
        out = dynamic_conv2d(out, generated_weights['up1_w'], padding=1)
        out = self.relu(self.in_up1(out))

        # Upscaling back to 256
        out = F.interpolate(out, scale_factor=2, mode='bilinear', align_corners=False)
        out = dynamic_conv2d(out, generated_weights['up2_w'], padding=1)
        out = self.relu(self.in_up2(out))       
        
        # Map back to 3 RGB channels
        # delta_raw = dynamic_conv2d(out, generated_weights['conv_out_w'], padding=4)
        delta_raw = self.conv_out(out)
        
        delta = torch.tanh(delta_raw)
        # delta = 0.75 * torch.tanh(delta_raw / 2.0)
        if debug:
            print(
                f"delta stats -> mean_abs: {delta.abs().mean().item():.5f}, "
                f"std: {delta.std().item():.5f}, "
                f"max_abs: {delta.abs().max().item():.5f}"
            )

        return content_img + delta

In [14]:
def train_weights(num_epochs=1, lr=1e-4, max_steps=20000, warmup_steps=200, grad_clip=1.0, content_weight = 0.5, 
                  style_weight = 10.0, tv_weight = 2e-2, kernel_tv_weight = 100.0, ckpt_smoothing=0.98, save_path="meta_network_trained.pth", snapshot_steps=1000, 
                  snapshot_style_path="StarryNight.jpg", snapshot_content_path="TestContent.jpg", snapshot_direct="training_snapshots",
                 resume_from=None, step_offset=0):
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(256),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    content_dataset = ImageFolder(root='./content_data', transform=transform)
    style_dataset = ImageFolder(root='./style_data', transform=transform)
    
    content_loader = DataLoader(content_dataset, batch_size=8, shuffle=True, drop_last=True)
    style_loader = DataLoader(style_dataset, batch_size=8, shuffle=True, drop_last=True)

    meta_net = MetaNetwork().to(device)
        
    if resume_from is not None:
        meta_net.load_state_dict(torch.load(resume_from, map_location=device))
        print(f"Resumed model weights from: {resume_from}")
    
    transform_net = TransformationNetwork().to(device)
    loss_net = VGGPerceptualLoss().to(device)

    # VGG is only used for the loss.
    loss_net.eval()

    # Do NOT train the transformation network directly.
    transform_net.eval()

    optimizer = torch.optim.Adam(meta_net.parameters(),lr=lr)

    snapshot_ready = False
    if snapshot_steps and snapshot_steps > 0:
        try:
            snap_style_img = load_image(snapshot_style_path).to(device)
            snap_content_img = load_image(snapshot_content_path).to(device)
            os.makedirs(snapshot_direct, exist_ok=True)
            snapshot_ready = True
        except FileNotFoundError:
            print(f"Snapshot images not found, skipping periodic review...")

    step = 0

    print("Content batches:", len(content_loader))
    print("Style batches:", len(style_loader))

    meta_net.train()
    best_loss = float("inf")

    smoothed_loss = None

    for epoch in range(num_epochs):
        style_iter = iter(style_loader)

        for content_imgs, _ in content_loader:
            if step >= max_steps:
                break
            try:
                style_imgs, _ = next(style_iter)
            except StopIteration:
                style_iter = iter(style_loader)
                style_imgs, _ = next(style_iter)

            content_imgs = content_imgs.to(device)
            style_imgs = style_imgs.to(device)

            batch_size = min(
                content_imgs.size(0),
                style_imgs.size(0)
            )

            content_imgs = content_imgs[:batch_size]
            style_imgs = style_imgs[:batch_size]

            optimizer.zero_grad(set_to_none=True)

            warmup_factor = min(1.0, (step + 1) / max(1, warmup_steps))
            for g in optimizer.param_groups:
                g["lr"] = lr * warmup_factor

            generated_weights = meta_net(style_imgs)

            stylized_imgs = transform_net(
                content_imgs,
                generated_weights
            )

            c_loss, s_loss, s_loss1, s_loss2, s_loss3, s_loss4 = loss_net(stylized_imgs, content_imgs, style_imgs)
            tv_loss = total_variation_loss(stylized_imgs)
            kernel_tv_loss = sum(kernel_smoothness_loss(w) for w in generated_weights.values()) / len(generated_weights)
            total_loss = (content_weight * c_loss + style_weight * s_loss + tv_weight * tv_loss + kernel_tv_weight * kernel_tv_loss)

            total_loss.backward()

            grad_norm = torch.nn.utils.clip_grad_norm_(
                meta_net.parameters(),
                max_norm=grad_clip
            )

            optimizer.step()

            # Delta saturation check to ensure the run can be killed early should a collapse occur.
            # Collapse in this case refers to the delta being close to the bound tanh for a long period.
            if step == 0 or (step + 1) % 200 == 0:
                with torch.no_grad():
                    _ = transform_net(content_imgs, generated_weights, debug=True)

            loss_val = total_loss.item()
            smoothed_loss = (
                loss_val if smoothed_loss is None
                else ckpt_smoothing * smoothed_loss + (1 - ckpt_smoothing) * loss_val
            )

            if smoothed_loss < best_loss:
                best_loss = smoothed_loss
        
                torch.save(
                    meta_net.state_dict(),
                    save_path
                )
        
                print(
                    f"  Saved best checkpoint (smoothed loss): "
                    f"{best_loss:.6f}"
                )
            global_step = step + step_offset

            if snapshot_ready and  (step == 0 or (step + 1) % snapshot_steps == 0):
                meta_net.eval()
                with torch.no_grad():
                    snap_weights = meta_net(snap_style_img)
                    snap_out = transform_net(snap_content_img, snap_weights)
                    save_image(snap_out, os.path.join(snapshot_direct, f"Step_{global_step + 1:05d}.jpg"))
                # torch.save(meta_net.state_dict(), os.path.join(snapshot_direct, f"step_{step + 1:05d}.pth"))
                meta_net.train()
    
            step += 1
            if step == 1 or step % 10 == 0:
                print(
                    f"Step {global_step:4d} | "
                    f"Loss {total_loss.item():.6f} | "
                    f"Content {c_loss.item():.6f} | "
                    f"Style {s_loss.item():.6f} | "
                    f"TV {tv_loss.item():.6f} | "
                    f"Grad {grad_norm:.6f} | "
                    f"Kernel Loss {kernel_tv_loss:.6f} (×{kernel_tv_weight}={kernel_tv_weight*kernel_tv_loss:.4f})"
                )

        if step >= max_steps:
            break

    final_path = save_path.replace(".pth", "_final.pth")
    torch.save(meta_net.state_dict(), final_path)
    
    print()
    print("Training complete.")
    print("Best model saved to meta_network_trained.pth")
    print(f"Final model saved to {final_path}")

    return meta_net

In [ ]:
# Standard output
train_weights(lr=1e-4, max_steps=40000, warmup_steps=200, style_weight=10.0, tv_weight=2e-2, kernel_tv_weight=150.0, snapshot_steps=500)

In [16]:
# Style weight value is required due to the normalization applied beofrehand in normalize_dynamic_weight
def run_style_transfer(style_path, content_path, output_path, model_checkpoint_path, content_weight=1.0, 
                       style_weight=1e4, meta_net=None, transform_net=None, losses_net=None, compute_metrics=True, exact_time=True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    t0 = time.perf_counter()
        
    # Load inputs
    style_img = load_image(style_path).to(device)
    content_img = load_image(content_path).to(device)
    t1 = time.perf_counter()

    if meta_net is None:
        meta_net = MetaNetwork().to(device)
    if transform_net is None:
        transform_net = TransformationNetwork().to(device)
    if losses_net is None and compute_metrics:
        losses_net = VGGPerceptualLoss().to(device)
    t2 = time.perf_counter()
    
    # Load trained model parameters θ
    meta_net.load_state_dict(torch.load(model_checkpoint_path, map_location=device))
    meta_net.eval()
    transform_net.eval()
    t3 = time.perf_counter()
    
    with torch.no_grad():
        generated_weights = meta_net(style_img)
        t4 = time.perf_counter()
        # Pass content image Ic through Transformation Network N_w
        output_tensor = transform_net(content_img, generated_weights, debug=True)

        # Compute and print losses - mostly for debugging purposes so set compute_metrics to false if you don't want to see it
        if compute_metrics:
            c_loss, s_loss, s_loss1, s_loss2, s_loss3, s_loss4 = losses_net(output_tensor, content_img, style_img)
            total_loss = (content_weight * c_loss) + (style_weight * s_loss)
            
            print("--- Neural Style Transfer Loss Metrics ---")
            print(f"Content Loss : {c_loss.item():.4f} (Weighted: {content_weight * c_loss.item():.4f})")
            print(f"Style Loss   : {s_loss.item():.4f} (Weighted: {style_weight * s_loss.item():.4f})")
            print(f"Total Loss   : {total_loss.item():.4f}")
            print("------------------------------------------")
        t5 = time.perf_counter()
    
    # Save resulting image to file system
    save_image(output_tensor, output_path)
    t6 = time.perf_counter()

    if exact_time:
        print(
            f"Time to load images: {t1-t0:.3f}s |",
            f"Building models: {t2-t1:.3f}s |",
            f"Loading checkpoints: {t3-t2:.3f}s |",
            f"Meta Network(forward): {t4-t3:.3f}s |",
            f"Metrics: {t5-t4:.3f}s |",
            f"Outputting Image: {t6-t5:.3f}s"
        )
    

    diff = (output_tensor - content_img).abs().mean(dim=1, keepdim=True)
    diff_normalized = diff / (diff.max() + 1e-8)
    heatmap_path = output_path.rsplit(".", 1)[0] + "_diff_heatmap.jpg"
    heatmap_uint8 = (diff_normalized.squeeze().clamp(0, 1) * 255).byte().cpu().numpy()
    Image.fromarray(heatmap_uint8, mode="L").save(heatmap_path)

# Example execution:
# start_time = time.perf_counter()
# run_style_transfer('StarryNight.jpg', 'TestContent.jpg', 'output_result.jpg', 'meta_network_trained_final.pth')
# end_time = time.perf_counter()

# Calculate elapsed time
# execution_time = end_time - start_time
# print(f"Execution time: {execution_time:.6f} seconds")

In [ ]:
# To be used for resumption of training from a checkpoint
train_weights(lr=1e-4, max_steps=40000, warmup_steps=200, style_weight=10.0, tv_weight=2e-2, kernel_tv_weight=150.0, snapshot_steps=500, resume_from="meta_network_trained_final.pth", step_offset=20000, save_path="meta_network_trained_step40000.pth")

In [ ]:
train_weights(lr=1e-4, max_steps=40000, warmup_steps=200, style_weight=10.0, tv_weight=2e-2, kernel_tv_weight=150.0, snapshot_steps=500, resume_from="meta_network_trained_step40000_final.pth", step_offset=40000, save_path="meta_network_trained_step60000.pth")

In [35]:
run_style_transfer('SunsetFires.jpg', 'ParkatNTU.jpeg', 'output_result_image3_final.jpg', 'meta_network_trained_step60000_final.pth')

delta stats -> mean_abs: 0.22715, std: 0.19755, max_abs: 0.92668
--- Neural Style Transfer Loss Metrics ---
Content Loss : 2.6318 (Weighted: 2.6318)
Style Loss   : 0.4513 (Weighted: 4513.0381)
Total Loss   : 4515.6699
------------------------------------------
Successfully saved stylized image to: output_result_image3_final.jpg
Time to load images: 0.072s | Building models: 4.080s | Loading checkpoints: 0.417s | Meta Network(forward): 0.010s | Metrics: 0.076s | Outputting Image: 0.003s


In [36]:
run_style_transfer('SunsetFires.jpg', 'RoadatNTU.jpeg', 'output_result_image4_final.jpg', 'meta_network_trained_step60000_final.pth')

delta stats -> mean_abs: 0.11753, std: 0.14717, max_abs: 0.68314
--- Neural Style Transfer Loss Metrics ---
Content Loss : 2.4786 (Weighted: 2.4786)
Style Loss   : 0.5196 (Weighted: 5196.1267)
Total Loss   : 5198.6050
------------------------------------------
Successfully saved stylized image to: output_result_image4_final.jpg
Time to load images: 0.062s | Building models: 3.995s | Loading checkpoints: 0.580s | Meta Network(forward): 0.011s | Metrics: 0.069s | Outputting Image: 0.003s


In [41]:
run_style_transfer('TheSailor.jpg', 'TestContent.jpg', 'output_result.jpg', 'meta_network_trained_step60000_final.pth')

delta stats -> mean_abs: 0.21342, std: 0.24565, max_abs: 0.88143
--- Neural Style Transfer Loss Metrics ---
Content Loss : 2.9316 (Weighted: 2.9316)
Style Loss   : 0.4701 (Weighted: 4701.1840)
Total Loss   : 4704.1157
------------------------------------------
Successfully saved stylized image to: output_result.jpg
Time to load images: 0.041s | Building models: 4.259s | Loading checkpoints: 0.546s | Meta Network(forward): 0.010s | Metrics: 0.079s | Outputting Image: 0.003s


In [43]:
run_style_transfer('TheSailor.jpg', 'TestImage2.jpg', 'output_result_image2_final.jpg', 'meta_network_trained_step60000_final.pth')

delta stats -> mean_abs: 0.06907, std: 0.09756, max_abs: 0.52873
--- Neural Style Transfer Loss Metrics ---
Content Loss : 1.0104 (Weighted: 1.0104)
Style Loss   : 0.5757 (Weighted: 5756.8419)
Total Loss   : 5757.8521
------------------------------------------
Successfully saved stylized image to: output_result_image2_final.jpg
Time to load images: 0.091s | Building models: 8.521s | Loading checkpoints: 1.803s | Meta Network(forward): 0.043s | Metrics: 0.211s | Outputting Image: 0.027s


In [39]:
run_style_transfer('TheSailor.jpg', 'ParkatNTU.jpeg', 'output_result_image3_final.jpg', 'meta_network_trained_step60000_final.pth')

delta stats -> mean_abs: 0.13683, std: 0.17770, max_abs: 0.90062
--- Neural Style Transfer Loss Metrics ---
Content Loss : 3.0177 (Weighted: 3.0177)
Style Loss   : 0.4992 (Weighted: 4992.1376)
Total Loss   : 4995.1553
------------------------------------------
Successfully saved stylized image to: output_result_image3_final.jpg
Time to load images: 0.052s | Building models: 4.195s | Loading checkpoints: 0.454s | Meta Network(forward): 0.011s | Metrics: 0.072s | Outputting Image: 0.003s


In [42]:
run_style_transfer('TheSailor.jpg', 'RoadatNTU.jpeg', 'output_result_image4_final.jpg', 'meta_network_trained_step60000_final.pth')

delta stats -> mean_abs: 0.12951, std: 0.15736, max_abs: 0.80860
--- Neural Style Transfer Loss Metrics ---
Content Loss : 2.4745 (Weighted: 2.4745)
Style Loss   : 0.3976 (Weighted: 3975.9895)
Total Loss   : 3978.4641
------------------------------------------
Successfully saved stylized image to: output_result_image4_final.jpg
Time to load images: 0.150s | Building models: 5.835s | Loading checkpoints: 1.181s | Meta Network(forward): 0.024s | Metrics: 0.236s | Outputting Image: 0.009s


In [ ]:
train_weights(lr=1e-4, max_steps=40000, warmup_steps=200, style_weight=10.0, tv_weight=2e-2, kernel_tv_weight=150.0, snapshot_steps=500, resume_from="meta_network_trained_step60000_final.pth", step_offset=60000, save_path="meta_network_trained_step80000.pth")